#Instructions

This code will add satellite features to your site labels using Google Earth Engine.

Before running this code, you must have a labels .csv file uploaded to your Google Earth Engine (GEE) project. See workflow document for instructions on how to use GEE.

For best results, use the first code for generating site labels.

This code is specifically for .csv files that do not have a Date column.

Before running this code, please scroll through and identify the locations that say "USER INPUT". There are multiple spots in the second block of cdoe where you must input your own file names and this code will not run correctly if you do not update these sections.

This code works for both state and federal labels.


#Important Note!
Once this code runs, it will generate a series of .csv files that save to the highest folder of your G Drive. This is just how GEE works, and there is unfortunately no way to direct these .csv files to a specific folder.

You will see these .csv files slowly populate in your G Drive folder. This code will say "complete" before those files populate, so after completing this code you'll have to wait a few extra minutes to make sure all files are there. There might be a lot, depending on how many labels you created. 1-2 .csv files typically populate per minute, so once you notice none have populated after a few minutes, you are ready to move onto the next step.


In [1]:
import ee
import numpy as np
import os
from matplotlib import pyplot as plt
import pickle
import pandas as pd
import numpy as np
import shapely
import geopandas as gpd
import ee # earth engine
import folium
import datetime

from shapely.geometry import Point
from google.colab import drive

DEBUG = False



#User Input Section
Fill out the following section with your parameters.

In [2]:
ee.Authenticate()

#------USER INPUT--------#
#Input your GEE project name below, between the ' '. The name can be found in the top righthand corner of your GEE interface.
ee.Initialize(project='superfund-sites-497816') #name here


#------USER INPUT--------#
# must change to .01/.001 depending on your preferred grid size resolution.
#This must match the grid size used to create your labels.
grid_delta = .01 #input resolution here
res: float = 0.01 #and here
buff = (res*5)/10

if res == 0.01:
  scale: float = 1000
elif res == 0.1:
  scale: float = 100
elif res == 0.001:
  scale: float = 10000 #this might be wrong, night be 5000? idk

bands = np.arange(0,64).astype(str)
bands = ["A0" + x if len(x)==1 else "A"+x for x in bands]

# Combined reducer. Processes each band that we want to summarize over the given rectangle
reducer = (ee.Reducer.mean()
           .combine(ee.Reducer.percentile([0,10,20,30,40,50,60,70,80,90,100]), '', True))

#------USER INPUT--------#
#set correct path to asset within the " ": see table ID in GEE interface by clicking on the appropriate asset
sparse_coords_ee_asset_name = "projects/superfund-sites-497816/assets/mn_remediation_combinedlabels_01_2026-06-24_1955"

#------USER INPUT--------#
#Input number of features below
#See "number of features" in the "asset details" box in GEE - same place where you found table ID
#AKA the total row count in your labels .csv file
total = 16932

In [3]:
points = ee.FeatureCollection(sparse_coords_ee_asset_name)
points = points.toList(points.size())


if DEBUG:
    points = points.slice(0,1_000)

def make_rect(feature):
    lon = ee.Number(feature.get('lon'))
    lat = ee.Number(feature.get('lat'))

    rect = ee.Geometry.Rectangle([
        lon.subtract(buff), lat.subtract(buff),
        lon.add(buff), lat.add(buff)
    ])

    return ee.Feature(rect, feature.toDictionary())

batch_size = 500  # features per batch
if DEBUG:
    total = 1_000
num_batches = (total // batch_size) + 1

print(f"Total features: {total}, batches: {num_batches}")

# Get today's date
today_date = datetime.date.today().strftime("%Y-%m-%d")

# Loop over batches and export each one
for i in range(num_batches):
    start = i * batch_size
    end = start + batch_size
    subset = points.slice(start,end)
    subset = ee.FeatureCollection(subset)
    fname = f'GEE_featurization_{today_date}_{i}'


    rects = subset.map(make_rect)

    collection = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
              .filterDate("2025-01-01") #Imagery is set to a specific year
              .filterBounds(rects)
             )

    composite = collection.select(bands).median() ## Should just be one image anyway. Median over time dimension

    stats_per_rect = composite.reduceRegions(
        collection=rects,
        reducer=reducer,
        scale=scale,
        crs='EPSG:4326',
    )

    ## Start download

    task = ee.batch.Export.table.toDrive(
        collection=stats_per_rect,
        description=f'GEE_MN_featurization_{today_date}_{i}', #OPTIONAL USER INPUT TO CHANGE FILE NAME
        fileNamePrefix=fname,
        fileFormat='CSV')

    task.start()
    print('Export task started:', task.status())


    #STOP HERE. FILES HAVE BEEN CREATED.
    #NOTE: Files will save to the highest folder in your Google Drive. This takes a few minutes (even after the code finishes running).
    #You may move them as you please.

Total features: 16932, batches: 34
Export task started: {'state': 'READY', 'description': 'GEE_MN_featurization_2026-07-16_0', 'priority': 100, 'creation_timestamp_ms': 1784222177723, 'update_timestamp_ms': 1784222177723, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'C43WH6WB67RFNIGBE5QQCUIM', 'name': 'projects/superfund-sites-497816/operations/C43WH6WB67RFNIGBE5QQCUIM'}
Export task started: {'state': 'READY', 'description': 'GEE_MN_featurization_2026-07-16_1', 'priority': 100, 'creation_timestamp_ms': 1784222179382, 'update_timestamp_ms': 1784222179382, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'RD6KQTQOHFUHSRIEOYO42IFX', 'name': 'projects/superfund-sites-497816/operations/RD6KQTQOHFUHSRIEOYO42IFX'}
Export task started: {'state': 'READY', 'description': 'GEE_MN_featurization_2026-07-16_2', 'priority': 100, 'creation_timestamp_ms': 1784222181022, 'update_timestamp_ms': 1784222181022, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': '